<a href="https://colab.research.google.com/github/gav-ip/ML-zero/blob/main/pytorch_of_transformer_addition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import random as py_random

USE_GPU = True
device = torch.device("cpu") if not USE_GPU or not torch.cuda.is_available() else torch.device("cuda")
block_size = 11
batch_size = 512 # Increased for potentially faster training, monitor memory
eval_iters = 200
max_iters = 1000 # Reduced for faster experimentation
eval_interval = 100 # Increased to reduce evaluation frequency
n_embed = 16 # Reduced for faster computation
n_heads = 2 # Reduced for faster computation
n_layer = 3 # Reduced for faster computation
dropout = 0.2
learning_rate = 1e-2 # Can be adjusted with batch_size, but requires tuning

print(device)

cuda


In [3]:
vocab = {0:'0', 1:'1', 2:'2', 3:'3', 4:'4', 5:'5', 6:'6', 7:'7', 8:'8', 9:'9', 10:'+', 11:'=', 12:' '}
vocab_size = len(vocab)

In [4]:
stoi = {vocab[i]:i for i, ch in enumerate(vocab)}
itos = {i:vocab[i] for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[int(i)] for i in l])

print(encode("8+2 =10 "))
print(decode(encode("8+7 =15 ")))

[8, 10, 2, 12, 11, 1, 0, 12]
8+7 =15 


In [5]:
!wget https://raw.githubusercontent.com/gav-ip/ML-zero/main/transformer-adder/addition.txt

--2026-11-20 15:03:59--  https://raw.githubusercontent.com/gav-ip/ML-zero/main/transformer-adder/addition.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1300000 (1.2M) [text/plain]
Saving to: ‘addition.txt’

addition.txt        100%[===================>]   1.24M  --.-KB/s    in 0.03s   

2026-11-20 15:03:59 (45.4 MB/s) - ‘addition.txt’ saved [1300000/1300000]



In [6]:
with open("addition.txt", "r", encoding="utf-8") as f:
    text = [line.rstrip("\n") for line in f]

# Encode each line individually and convert to a PyTorch tensor
data = torch.tensor([encode(line) for line in text], dtype=torch.long, device=device)
data[:3]

tensor([[ 8,  5,  0, 10,  6,  6,  0, 11,  0,  1,  5,  1],
        [ 2,  2,  7, 10,  8,  5,  7, 11,  4,  8,  0,  1],
        [ 7,  1,  7, 10,  5,  4,  6, 11,  3,  6,  2,  1]], device='cuda:0')

In [7]:
print(data[:5])
print(decode(data[0]))

tensor([[ 8,  5,  0, 10,  6,  6,  0, 11,  0,  1,  5,  1],
        [ 2,  2,  7, 10,  8,  5,  7, 11,  4,  8,  0,  1],
        [ 7,  1,  7, 10,  5,  4,  6, 11,  3,  6,  2,  1],
        [ 7,  4,  0, 10,  9,  1,  4, 11,  4,  5,  6,  1],
        [ 3,  7,  8, 10,  2,  5,  2, 11,  0,  3,  6, 12]], device='cuda:0')
850+660=0151


In [8]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split), (batch_size,))
    seq = data_split[ix]

    x = seq[:, :]
    y = seq[:, 1:].clone()
    y[:, :7] = -1

    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(decode(xb[0].tolist()))
print('targets:')
print(yb.shape)
print(yb[0].tolist())

inputs:
torch.Size([512, 12])
756+201=759 
targets:
torch.Size([512, 11])
[-1, -1, -1, -1, -1, -1, -1, 7, 5, 9, 12]


In [10]:
class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size + 1, block_size + 1)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        # Compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * (C**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        # Perform the weighted aggregation of the values
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ff = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class AdditionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size + 1, n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed, n_heads) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            valid = targets != -1
            loss = F.cross_entropy(logits, targets, ignore_index=-1)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [11]:
model = AdditionModel().to(device)

In [12]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
         losses = torch.zeros(eval_iters)
         for k in range(eval_iters):
             X, Y = get_batch(split)
             _, loss = model(X, Y)
             losses[k] = loss.item()
         out[split] = losses.mean().item()
    model.train()
    return out

In [13]:
param_count = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {param_count}")

Total parameters: 10349


In [14]:
# We can fix the get_batch function to align sequence lengths.
# Let's override get_batch so both inputs and targets have a sequence length of 11.
def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split), (batch_size,))
    seq = data_split[ix]

    # Slice both to sequence length of 11
    x = seq[:, :-1]
    y = seq[:, 1:].clone()
    y[:, :7] = -1

    return x.to(device), y.to(device)

# Initialize the optimizer and run training
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for steps in range(max_iters):
    if steps % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # get sample batch from training set
    xb, yb = get_batch('train')

    # Evaluate loss and compute gradients
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 2.7847, val loss 2.7819
step 100: train loss 1.7361, val loss 1.7372
step 200: train loss 1.7191, val loss 1.7184
step 300: train loss 1.4813, val loss 1.4800
step 400: train loss 1.0806, val loss 1.0817
step 500: train loss 0.9106, val loss 0.9112
step 600: train loss 0.8125, val loss 0.8131
step 700: train loss 0.7656, val loss 0.7659
step 800: train loss 0.7368, val loss 0.7370
step 900: train loss 0.7275, val loss 0.7293


In [16]:
import pickle

checkpoint = {
    'model_state_dict': model.state_dict(),
    'stoi': stoi,
    'itos': itos,
    'config': {
        'vocab_size': vocab_size,
        'block_size': block_size,
        'n_embd': n_embed,
        'n_head': n_heads,
        'n_layer': n_layer,
        'dropout': dropout,
    }
}
with open('addition_model.pkl', 'wb') as f:
    pickle.dump(checkpoint, f)

In [17]:
with open('addition_model.pkl', 'rb') as f:
    ckpt = pickle.load(f)

cfg = ckpt['config']
stoi, itos = ckpt['stoi'], ckpt['itos']

m = AdditionModel().to(device)
m.load_state_dict(ckpt['model_state_dict'])
m.eval()

AdditionModel(
  (token_embedding_table): Embedding(13, 16)
  (position_embedding_table): Embedding(12, 16)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-1): 2 x Head(
            (key): Linear(in_features=16, out_features=8, bias=False)
            (query): Linear(in_features=16, out_features=8, bias=False)
            (value): Linear(in_features=16, out_features=8, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=16, out_features=16, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ff): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=16, out_features=64, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=64, out_features=16, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((16,), eps=1e-05, elementwise_affine=True

In [50]:
prompt = "499+213="
idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
out = m.generate(idx, max_new_tokens=4)
result = decode(out[0].tolist())
res_start = result[:-4]
sum = result[-4:]
reversed_res = res_start + sum[::-1]

print(result)
print(reversed_res)

499+213=276 
499+213= 672


In [40]:
# Example of a BTC (Batch, Time, Channel) tensor
# Let's assume B (Batch) = 2, T (Time / Sequence) = 3, C (Channel / Embedding) = 4
B, T, C = 2, 3, 4
dummy_btc_tensor = torch.arange(B * T * C, dtype=torch.float32).view(B, T, C)

print("=== BTC Tensor Shape ===")
print(dummy_btc_tensor.shape)  # Should show [2, 3, 4] -> [B, T, C]
print("\n=== BTC Tensor Content ===")
for b in range(B):
    print(f"Batch item {b}:")
    for t in range(T):
        print(f"  Time step {t} (vector of length C={C}): {dummy_btc_tensor[b, t].tolist()}")

=== BTC Tensor Shape ===
torch.Size([2, 3, 4])

=== BTC Tensor Content ===
Batch item 0:
  Time step 0 (vector of length C=4): [0.0, 1.0, 2.0, 3.0]
  Time step 1 (vector of length C=4): [4.0, 5.0, 6.0, 7.0]
  Time step 2 (vector of length C=4): [8.0, 9.0, 10.0, 11.0]
Batch item 1:
  Time step 0 (vector of length C=4): [12.0, 13.0, 14.0, 15.0]
  Time step 1 (vector of length C=4): [16.0, 17.0, 18.0, 19.0]
  Time step 2 (vector of length C=4): [20.0, 21.0, 22.0, 23.0]


In [49]:
def evaluate_exact_match_accuracy(model, dataset, num_samples=1000):
    model.eval()
    correct = 0
    total = 0

    # Select a random subset of the dataset for validation evaluation
    indices = torch.randint(len(dataset), (num_samples,))
    samples = dataset[indices]

    for i in range(num_samples):
        seq = samples[i].tolist()
        decoded_seq = decode(seq)

        # Split into prompt (e.g., "499+213=") and target (e.g., "0151" or "277 ")
        prompt, target_sum = decoded_seq.split('=')
        prompt_encoded = torch.tensor([encode(prompt + '=')], dtype=torch.long, device=device)

        # Generate tokens (length of target sum)
        with torch.no_grad():
            generated_out = model.generate(prompt_encoded, max_new_tokens=len(target_sum))

        generated_str = decode(generated_out[0].tolist())
        predicted_sum = generated_str[len(prompt)+1:]

        # Clean up whitespaces and compare reversed outputs (if the dataset used reversed addition targets)
        # Let's inspect raw text formats. In this dataset, the sums are stored in reverse order
        # (e.g., 850+660=0151 which is 1510 reversed). Let's compare the raw predicted string with the target string.
        if predicted_sum.strip() == target_sum.strip():
            correct += 1
        total += 1

    accuracy = (correct / total) * 100
    print(f"Evaluated on {num_samples} validation samples.")
    print(f"Exact Match (EM) Accuracy: {accuracy:.2f}%")
    return accuracy

# Run evaluation on the validation dataset
evaluate_exact_match_accuracy(m, val_data, num_samples=500)

Evaluated on 500 validation samples.
Exact Match (EM) Accuracy: 7.60%


7.6